# IEMOCAP Preprocessing Script
This script is used for generating IEMOCAP_RAW_PREPROCESSED data from the raw IEMOCAP data, you can download the raw dataset from:


In [1]:
import os, sys
import glob
import pickle
import numpy as np
import pandas as pd
import cv2
from scipy.io import wavfile
from tqdm import tqdm
import torch
from facenet_pytorch import MTCNN

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=48, margin=2, post_process=False, device=device)

# Common Functions

In [3]:
def read_video(file_name):
    vidcap = cv2.VideoCapture(file_name)
    
    # Read FPS
    (major_ver, minor_ver, subminor_ver) = (cv2.__version__).split('.')
    if int(major_ver)  < 3 :
        fps = vidcap.get(cv2.cv.CV_CAP_PROP_FPS)
    else :
        fps = vidcap.get(cv2.CAP_PROP_FPS)
    
    # Read image data
    success, image = vidcap.read()
    images = []
    while success:
        images.append(image)
        success, image = vidcap.read()
    return np.stack(images), fps

def parse_evaluation_transcript(eval_lines, transcript_lines):
    metadata = {}
    
    # Parse Evaluation
    for line in eval_lines:
        if line.startswith('['):
            tokens = line.strip().split('\t')
            time_tokens = tokens[0][1:-1].split(' ')
            start_time, end_time = float(time_tokens[0]), float(time_tokens[2])
            uttr_id, label = tokens[1], tokens[2]
            metadata[uttr_id] = {'start_time': start_time, 'end_time': end_time, 'label': label}

    # Parse Transcript
    trans = []
    for line in transcript_lines:
        tokens = line.split(':')
        uttr_id = tokens[0].split(' ')[0]
        if '_' not in uttr_id:
            continue
        text = tokens[-1].strip()
        try:
            metadata[uttr_id]['text'] = text
        except KeyError:
            print(f'KeyError: {uttr_id}')
    return metadata

def retrieve_audio(signal, sr, start_time, end_time):
    start_idx = int(sr * start_time)
    end_idx = int(sr * end_time)
    audio_segment = signal[start_idx:end_idx]
    return audio_segment, sr

def retrieve_video(frames, fps, start_time, end_time):
    start_idx = int(fps * start_time)
    end_idx = int(fps * end_time)
    images = frames[start_idx:end_idx,:,:,:]
    return images, fps

def crop_img_center(img: torch.tensor, target_size=48):
    '''
    Some images have un-detectable faces,
    to make the training goes normally,
    for those images, we crop the center part,
    which highly likely contains the face or part of the face.

    @img - (channel, height, width)
    '''
    current_size = img.size(1)
    off = (current_size - target_size) // 2 # offset
    cropped = img[:, off:off + target_size, off - target_size // 2:off + target_size // 2]
    return cropped

def dump_image_audio(uttr_id, audio_segment, sr, img_segment, img_segment_L, img_segment_R, fps, out_path='./', grayscale=False):
    out_path = f'{out_path}/{"_".join(uttr_id.split("_")[:2])}'
    if not os.path.exists(f'{out_path}/{uttr_id}'):
        os.makedirs(f'{out_path}/{uttr_id}')
    wavfile.write(f'{out_path}/{uttr_id}/audio.wav', sr, audio_segment)
    wavfile.write(f'{out_path}/{uttr_id}/audio_L.wav', sr, audio_segment[:,0])
    wavfile.write(f'{out_path}/{uttr_id}/audio_R.wav', sr, audio_segment[:,1])    
    for i in range(img_segment.shape[0]):
#         cv2.imwrite(f'./{out_path}/{uttr_id}/image_{i}.jpg', img_segment[i,:,:,:])
        imgL = img_segment_L[i,:,:,:]
        imgR = img_segment_R[i,:,:,:]
        if grayscale:
            imgL = rgb2gray(imgL)
            imgR = rgb2gray(imgR)
            
        faceL = mtcnn(imgL)
        if faceL == None:
            faceL = crop_img_center(torch.tensor(imgL).permute(2, 0, 1))
        cv2.imwrite(f'{out_path}/{uttr_id}/image_L_{i}.jpg', faceL.permute(1, 2, 0).int().numpy())
        faceR = mtcnn(imgR)
        if faceR == None:
            faceR = crop_img_center(torch.tensor(imgR).permute(2, 0, 1))
        cv2.imwrite(f'{out_path}/{uttr_id}/image_R_{i}.jpg', faceR.permute(1, 2, 0).int().numpy())
    
#         cv2.imwrite(f'{out_path}/{uttr_id}/image_L_{i}.jpg', imgL)
#         cv2.imwrite(f'{out_path}/{uttr_id}/image_R_{i}.jpg', imgR)

def rgb2gray(rgb):
    r, g, b = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    return gray

def crop(imgs, target_size=224):
    # imgs.shape = (180, 480, 360, 3)
    _, h, w, _ = imgs.shape
    offset_h = (h - target_size) // 2
    offset_w = (w - target_size) // 2
    imgs = imgs[:, offset_h:-offset_h, offset_w:-offset_w, :]
    return imgs

In [4]:
%%time
# Process multimodal data over all sessions
# NOTE: This might take several hours to run, the time listed on this cell is for processing 5 label files
output_path = r'/home/matt/Model/Database_processed/IEMOCAP_RAW_PROCESSED_Face'

if not os.path.exists(output_path):
    os.makedirs(output_path)
    
all_metas = {}
for base_path in glob.glob(r'/home/matt/Model/Data/IEMOCAP_full_release/IEMOCAP_full_release/Session*'):
    avi_path = f'{base_path}/dialog/avi/DivX'
    script_path = f'{base_path}/dialog/transcriptions'
    wav_path = f'{base_path}/dialog/wav'
    label_path = f'{base_path}/dialog/EmoEvaluation/'
        
    for eval_fname in tqdm(glob.glob(f'{label_path}/*.txt')):
        avi_fname = f'{avi_path}/{eval_fname.split("/")[-1].replace(".txt", ".avi")}'
        wav_fname = f'{wav_path}/{eval_fname.split("/")[-1].replace(".txt", ".wav")}'
        script_fname = f'{script_path}/{eval_fname.split("/")[-1]}'

        eval_lines = open(eval_fname).readlines()
        transcript_lines = open(script_fname).readlines()
        sr, signal  = wavfile.read(wav_fname)

        images, fps = read_video(avi_fname)

        # Retrieve uttr_id, label, time, and transcript
        metas = parse_evaluation_transcript(eval_lines, transcript_lines)

        for uttr_id, metadata in metas.items():
            # Retrieve and Store Audio
            audio_segment, sr = retrieve_audio(signal, sr, metadata['start_time'], metadata['end_time'])
            metadata['sr'] = sr

            img_segment, fps = retrieve_video(images, fps, metadata['start_time'], metadata['end_time'])  
            img_segment_L, img_segment_R = img_segment[:,:,:img_segment.shape[2] // 2,:], img_segment[:,:,img_segment.shape[2] // 2:,:]
            img_segment_L = crop(img_segment_L)
            img_segment_R = crop(img_segment_R)
            metadata['fps'] = fps

            dump_image_audio(uttr_id, audio_segment, sr, img_segment, img_segment_L, img_segment_R, fps, out_path=output_path)

        # Update all metas
        all_metas.update(metas)
pickle.dump(all_metas, open(f'{output_path}/meta.pkl','wb'))

 12%|█████▏                                   | 4/32 [12:20<1:43:52, 222.60s/it]

KeyError: Ses03M_impro05b_FXX0
KeyError: Ses03M_impro05b_FXX1
KeyError: Ses03M_impro05b_FXX2


 28%|███████████▌                             | 9/32 [30:47<1:28:08, 229.95s/it]

KeyError: Ses03M_impro06_MXX0


 31%|████████████▌                           | 10/32 [33:15<1:15:03, 204.70s/it]

...

KeyError: Ses05F_impro02_FXX0
KeyError: Ses05F_impro02_MXX0


 97%|██████████████████████████████████████▋ | 30/31 [1:46:01<04:05, 245.94s/it]

KeyError: Ses05M_impro03_FXX0
KeyError: Ses05M_impro03_FXX0
KeyError: Ses05M_impro03_FXX1


100%|████████████████████████████████████████| 30/30 [1:30:09<00:00, 180.30s/it]

CPU times: user 22h 38min 55s, sys: 31min 51s, total: 23h 10min 46s
Wall time: 8h 24min 37s
